# 作业三：构建NDArray库

在本作业中，你将构建一个支撑大多数深度学习系统处理过程的基础库：n维数组（即NDArray）。到目前为止，你们主要使用numpy来实现这一目的，但本作业将引导你开发属于自己的（尽管功能更为有限）numpy变体，该库将同时支持CPU和GPU后端。更重要的是，与numpy（甚至PyTorch等变体）不同，你不会直接调用现有的高度优化的矩阵乘法或其他操作代码，而是实际编写自己版本的代码，这些版本在性能上能与支撑这些标准库的高度优化代码相媲美（按某些标准衡量，即"仅慢2-3倍"...这比可能轻易慢100倍的简单代码要好得多）。这个类最终将被集成到`needle`中，但在本次作业中你_只需_专注于ndarray模块，因为这是测试的唯一主题。

**注意**：为避免在Colab中耗尽有限的GPU资源，请先使用CPU运行时来编写和测试非GPU函数。在测试CUDA或GPU加速代码时再切换到GPU运行时。这种方法能确保GPU的高效使用，并防止在关键任务期间耗尽资源。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw3.git
%cd /content/drive/MyDrive/10714/hw3

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [ ]:
!make

make命令会读取当前目录下的Makefile文件。Makefile包含定义如何构建目标（如可执行文件或库）的规则。对于Makefile中指定的每个目标，make会检查目标文件及其依赖项（如.c、.cpp或.h文件）的时间戳。如果任何依赖项最近被修改过，make就会重新构建该目标。

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## 熟悉 NDArray 类

在开始本次作业时，你首先需要熟悉我们提供的 `NDArray.py` 类，这是本次作业的起点代码。该代码相当简洁（约500行，其中很多是你将要实现函数的注释说明）。

NDArray 类的核心是一个用于处理通用 n 维数组操作的 Python 封装器。需要记住的是，几乎所有这类数组在内部都是以浮点值向量的形式存储的，即：

```c++
float data[size];
```

而对数组不同维度的实际访问都是通过额外字段（如数组形状、步长等）来处理的，这些字段指示了这个"扁平"数组如何映射到 n 维结构。为了达到合理的运行速度，"原始"操作（如加法、二元运算，以及更结构化的操作如矩阵乘法等）都需要在某种原生语言（如 C++）中编写底层实现（包括调用 CUDA 等）。但是大量操作，如矩阵转置、广播、子集选取等，都可以仅通过调整数组的高层结构（如步长）来完成。

NDArray 类的设计理念是：我们希望处理数组结构的所有逻辑都用 Python 编写。只有真正在扁平向量上执行原始底层操作的"真正"低级代码（以及管理这些扁平向量的代码，因为它们可能需要分配在 GPU 上）是用 C++ 编写的。这种分离的具体性质在你完成作业的过程中会逐渐清晰起来，但总的来说，所有能在 Python 中完成的操作都在 Python 中完成；通常，这可能会以一些效率为代价……我们频繁调用 `.compact()`（这会复制内存）以使底层 C++ 实现更简单。

更详细地说，NDArray 类中有五个字段你需要熟悉（注意实际类成员这些字段都以下划线开头，例如 `_handle`、`_strides` 等，其中一些随后会作为公共属性暴露出来……在你的所有代码中，使用内部带下划线的版本即可）。

1. `device` - 一个 `BackendDevice` 类型的对象，它是一个简单的封装器，包含指向底层设备后端（如 CPU 或 CUDA）的链接。
2. `handle` - 一个存储数组底层内存的类对象。它被分配为 `device.Array()` 类型的类，不过这个分配过程都在提供的代码中完成（具体在 `NDArray.make` 函数中），你无需自己调用它。
3. `shape` - 一个元组，指定数组中每个维度的大小。
4. `strides` - 一个元组，指定数组中每个维度的步长。
5. `offset` - 一个整数，指示数组实际在底层 `device.Array` 内存中的起始位置（存储这个值很方便，这样我们可以更容易地管理指向现有内存的操作，而无需跟踪内存分配）。

通过操作这些字段，即使是纯 Python 代码也能执行许多所需的数组操作，例如维度置换（即转置）、广播等。而对于原始数组操作调用，`device` 类有许多方法（在 C++ 中实现）包含了必要的实现。

有几点需要注意：

* 在内部，该类可以使用任何高效的方式来操作数据数组作为"设备"后端。例如，甚至可以使用 numpy 数组，但我们不是用 `numpy.ndarray` 来表示 n 维数组，而是用 numpy 表示一个"扁平"的一维数组，然后调用相关的 numpy 方法来在这个原始内存上实现所有需要的操作符。这正是我们在 `ndarray_backend_numpy.py` 文件中所做的，它本质上提供了一个"存根参考"，仅使用 numpy 处理所有事情。你可以使用这个类来帮助你更好地调试自己为"原生" CPU 和 GPU 后端编写的"真实"实现。
* 对于你的许多 Python 实现来说，特别重要的是 `NDArray.make` 调用：
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
它会创建一个具有给定形状、步长、设备、句柄和偏移量的新 NDArray。如果未指定 `handle`（即未引用预先存在的内存），则调用将分配所需的内存；但如果指定了 handle，则不会分配新内存，新的 NDArray 指向与旧数组相同的内存。对于高效实现来说，尽可能多的函数不分配新内存非常重要，因此在许多情况下你将需要使用这个调用来实现这一点。
* NDArray 有一个 `.numpy()` 调用，可将数组转换为 numpy 数组。这与 "numpy_device" 后端不同：这会创建一个实际的 `numpy.ndarray`，它与给定的 NDArray 等价，即具有相同的维度、形状等，但不一定具有相同的步长（Pybind11 会以这种方式为返回的矩阵重新分配内存，这可能会改变步长模式）。

## 第一部分：Python 数组操作

作为你类的起点，请在 `ndarray.py` 文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

这些函数的输入/输出都在函数存根的文档字符串中有描述。需要强调的是，这些函数都不应该重新分配内存，而是应该返回与 `self` 共享相同内存的 NDArray，仅通过巧妙操作形状/步长等来获得必要的变换。

你可以参考课堂幻灯片中关于转置、广播和切片操作的提示来开始。

需要注意的一点是，与 numpy 不同，`__getitem__()` 调用永远不会改变数组的维度数。因此，对于二维 NDArray `A`，`A[2,2]` 将返回一个仍然是二维的数组，但只有一行一列。而 `A[:4,2]` 将返回一个具有 4 行 1 列的二维 NDArray。

在本节的所有代码中，你可以依赖 `ndarray_backend_numpy.py` 模块。你也可以查看等效 numpy 操作的结果（测试用例应该能说明这些操作是什么）。

实现这些函数后，你应该通过/提交以下测试。请注意，我们在下面的测试中测试了所有这四个函数，你可以在实现每个附加函数时逐步尝试通过更多测试。

In [ ]:
!python3 -m pytest -v -k "(permute or reshape or broadcast or getitem) and cpu and not compact"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

## 第二部分：CPU 后端 - Compact 和 setitem

在 `ndarray_backend_cpu.cc` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

要理解为什么这些函数属于同一类别，让我们考虑 `Compact()` 函数的实现。回忆一下，如果一个矩阵以"行优先"形式（实际上是行优先在高维数组上的推广）在内存中顺序布局，即最后一个维度在前，倒数第二个维度其次，依此类推直到第一个维度，那么该矩阵被认为是紧凑的。在我们的实现中，我们还要求分配的后端数组的总大小等于数组的大小（即底层数组在数组数据前后不能有任何数据，这意味着 `.offset` 字段等于零）。

现在让我们以 3D 数组为例，考虑紧凑调用可能如何工作。这里的 `shape` 和 `strides` 是被压缩矩阵的形状和步长（即在我们压缩它之前）。

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```
换句话说，我们正在从基于步长的矩阵表示转换为纯顺序的表示。

现在，实现 `Compact()` 的挑战在于你希望该方法适用于任意数量的输入维度。为不同的固定维度大小的数组专门处理很容易，但对于通用实现，你需要思考如何执行相同的操作，实际上你需要一个"可变数量的 for 循环"。作为提示，一种方法是维护一个索引向量（大小等于维度数），然后在循环中手动递增它们（包括当任何一个达到其最大大小时执行"进位"操作）。

但是，如果你在这方面真的遇到困难，你可以利用我们可能不会要求你处理超过 6 维的矩阵这一事实（尽管我们*会*使用 6 维，用于我们在课堂上讨论的 im2col 操作）。

#### 与 setitem 的连接

setitem 功能虽然看起来完全不同，但实际上与 `Compact()` 密切相关。`__setitem()__` 是在设置对象的某些元素时调用的 Python 函数，例如：
```python
A[::2,4:5,9] = 0 # 或 = some_other_array
```
你会如何实现这个？在上面的 `__getitem()__` 调用中，你已经实现了一个在不复制的情况下获取矩阵子集的方法（只是修改步长）。但是你会如何实际*设置*这个数组的元素？在这个作业的几乎所有其他设置中，我们在设置输出数组中的项目之前都会调用 `.compact()`，但在这里不行：调用 `.compact()` 会将子集数组复制到某个新内存，但 `__setitem__()` 调用的整个重点是我们想要修改现有内存。

如果你思考一会儿，你会意识到答案看起来很像 `.compact()` 但方向相反。如果我们想将一个（本身已经紧凑的）右侧矩阵分配给 `__getitem()__` 的结果，那么我们需要让 `shape` 和 `stride` 成为*输出*矩阵的这些字段。然后我们可以如下实现 setitem 调用：

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // 或 "= val;"
```
由于这种相似性，如果你以模块化的方式实现索引策略，你将能够在 `Compact()` 调用与 `EwiseSetitem()` 和 `ScalarSetitem()` 调用之间重用代码。

**注意**：在执行测试之前不要忘记运行 make 来重新构建你修改的 C++ 代码。

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## 第三部分：CPU后端 - 逐元素和标量运算

在`ndarray_backend_cpu.cc`文件中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

你可以参考已包含的`EwiseAdd()`和`ScalarAdd()`函数（以及`NDArray`中的调用方式）来理解这些函数所需的格式。

请注意，与这里提到的其他函数不同，我们没有为每个函数提供函数存根。这是因为虽然你可以通过单独实现每个函数来简单实现这些功能，但这会导致大量重复代码。你可以使用例如C++模板或宏来解决这个问题（但这些仅在内部暴露，不对外部接口公开）。

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## 第四部分：CPU后端 - 归约运算

在`ndarray_backend_cpu.cc`文件中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

一般来说，NDArray中的归约函数`.max()`和`.sum()`会沿着`axis`参数指定的轴（或者当`axis=None`时在整个数组上）计算最大值或求和；请注意我们不支持`axis`为轴集合的情况，尽管如果你想要添加这个功能也不会太难（但这不在你应该为作业实现的接口中）。

因为即使对于紧凑数组，沿着单个轴求和也可能有些棘手，这些Python函数通过将最后一个轴置换为要归约的轴（这是`NDArray`中`reduce_view_out()`函数的作用），然后压缩数组来简化操作。因此，对于你在C++中实现的`ReduceMax()`和`ReduceSum()`函数，你可以假设输入和输出数组在内存中都是连续的，并且你只需要对传递给C++函数的`reduce_size`大小的连续元素进行归约。

In [ ]:
!python3 -m pytest -v -k "reduce and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## Part 5: CPU Backend - Matrix multiplication

Implement the following functions in `ndarray_backend_cpu.cc`:

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

The first implementation, `Matmul()` can use the naive three-nested-for-loops algorithm for matrix multiplication.  However, the `MatmulTiled()` performs the same matrix multiplication on memory laid out in tiled form, i.e., as a contiguous 4D array
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

Make sure to initialize the output matrix to zero before populating it with the correct values during matrix multiplication.

Note that the Python `__matmul__` code already does the conversion to tiled form when all sizes of the matrix multiplication are divisible by `TILE`, so your code just needs to implement the multiplication in this form.  In order to make the methods efficient, you will want to make use of (after you implement it), the `AlignedDot()` function, which will enable the compiler to efficiently make use of vector operations and proper caching.  The output matrix will also be in the tiled form above, and the Python backend will take care of the conversion to a normal 2D array.

Note that in order to get the most speedup possible from you tiled version, you may want to use the clang compiler with colab instead of gcc.  To do this, run the following command before building your code.

In [ ]:
!python3 -m pytest -v -k "matmul and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

In [ ]:
!export CXX=/usr/bin/clang++ && make

## 第六部分：CUDA后端 - 紧凑化和赋值操作

在`ndarray_backend_cuda.cu`文件中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

在这一部分，你将在CUDA后端实现紧凑化和赋值调用。这与C++版本相当相似，但是根据你实现该函数的方式，也可能存在一些显著差异。不过，我们特别想强调C++和CUDA实现之间的几个区别。

首先，与CUDA后端代码中实现的示例函数一样，对于上述所有函数，你实际上需要实现两个函数：将从Python调用的上述基本函数，以及实际执行计算的相应CUDA内核。在大多数情况下，我们只在`ndarray_backend_cuda.cu`文件中提供了"基础"函数的原型，你需要自己定义和实现内核函数。但是，为了了解这些函数的工作原理，对于`Compact()`调用，我们为你提供了完整的`Compact()`调用和`CompactKernel()`调用的函数原型。

你可能会注意到一个看似奇怪的`CudaVec`结构体的使用，这是一个用于传递形状/步长参数的结构体。在C++版本中，我们使用STL的`std::vector`变量来存储这些输入（在基础的`Compact()`调用中也是如此），但CUDA内核无法操作STL向量，因此需要其他解决方案。此外，虽然我们可以将向量转换为普通的CUDA数组，但这会相当繁琐，因为我们需要调用`cudaMalloc()`，将参数作为整数指针传递，然后在调用后释放它们。当然，对于数组中实际的基础数据，这样的内存管理是必要的，但仅仅为了传递可变大小的小型形状/步长值元组而这样做似乎有些过度。解决方案是创建一个结构体，该结构体具有数组可以拥有的维度的"最大"大小，然后将实际的形状/步长数据存储在这些字段的前几个条目中。所有这些都由包含的`CudaVec`结构体和`VecToCuda()`函数完成，你可以在所有需要将形状/步长传递给内核本身的CUDA内核中直接使用这些提供的功能。

`Compact()`的C++和CUDA实现之间的另一个（更概念性的）重大区别是，在C++中，你通常会顺序遍历非紧凑数组的元素，这允许你在计算紧凑和非紧凑数组之间的对应索引时执行一些优化。在CUDA中，你无法这样做，需要实现能够直接从紧凑数组中的索引映射到跨步数组中的索引的代码。

和之前一样，我们建议你以这样的方式实现代码，使其可以轻松地在`Compact()`和`Setitem()`调用之间重用。简单提醒一下，如果你想从内核代码调用一个（独立的、非内核）函数，需要将其定义为`__device__`函数。在CUDA中，`__global__`用于定义在GPU上运行并从CPU主机调用的函数，而`__device__`定义仅在GPU上运行并从其他GPU函数代码调用的函数。

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## 第七部分：CUDA后端 - 逐元素和标量运算

在`ndarray_backend_cuda.cu`文件中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

同样，我们不提供这些函数的原型，你可以使用C++模板或宏来使实现更加紧凑。在实现每个函数后，你还需要取消注释Pybind11代码中的相应区域。

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## 第八部分：CUDA后端 - 归约运算

在`ndarray_backend_cuda.cu`文件中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

你可以采用相当简单的方法，只需为每个独立的归约项使用单独的CUDA线程：例如，如果有一个100×20的数组要在第二个维度上进行归约，你可以使用100个线程，每个线程独立处理自己的20维数组。这对于`.max(axis=None)`调用来说效率特别低，但我们暂时不担心这个问题。如果你想要一个更工业级的实现，可以使用分层机制，首先在较小的跨度上进行聚合，然后使用辅助函数在这些已归约的数组之间进行聚合，等等。但通过测试并不需要这样做。

In [ ]:
!python3 -m pytest -v -k "reduce and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## 第九部分：CUDA后端 - 矩阵乘法

在`ndarray_backend_cuda.cu`文件中实现以下函数：

* `Matmul()`

最后，作为你的最终练习，你将在GPU上实现矩阵乘法。你的实现可以大致遵循课堂上的讲解。虽然你可以使用相当朴素的代码通过测试（即只需为矩阵中的每个(i,j)位置分配一个单独的线程），但要高效地进行矩阵乘法（使其实际比CPU版本更快）需要协作获取和课堂中介绍的块共享内存寄存器平铺技术。尝试使用这些方法实现，看看你的代码能比C++（或numpy）后端快多少。

In [ ]:
!python3 -m pytest -v -k "matmul and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"